In [1]:
# Connect to postgis using geopandas
import warnings
import psycopg2
import pandas as pd
import geopandas as gpd
from shapely.geometry import MultiPolygon, Polygon
warnings.simplefilter(action='ignore', category=UserWarning)

con = psycopg2.connect(database="time", user="postgres", host="localhost")

In [2]:
# Read data
ilots_verniquet = gpd.read_postgis("select * from ilots_verniquet", con, geom_col='geom', index_col='gid')
ilots_vasserot = gpd.read_postgis("select * from ilots_vasserot", con, geom_col='geom', index_col='gid')
ilots_apur = gpd.read_postgis("select * from ilots_apur", con, geom_col='geom', index_col='gid')

In [3]:
# Define a function that tests if two geometries intersect
def match(geom1, geom2, threshold1=100, threshold2=0.2):
    if geom1.intersects(geom2):
        if geom1.intersection(geom2).area > threshold1 / 2:
            if geom1.intersection(geom2).area > threshold2 * min(geom1.area, geom2.area):
                return True
    return False

In [4]:
matchingLayer = gpd.GeoDataFrame(columns=['geometry', 'matching_id'])
count = 0

for geom1 in ilots_apur.geometry[:100]:
    for geom2 in ilots_vasserot.geometry[:100]:
        if match(geom1, geom2):
            count += 1
            # Append geom1 and geom2 to matchingLayer
            matchingLayer = gpd.GeoDataFrame(pd.concat([matchingLayer, gpd.GeoDataFrame({ 'matching_id': [count, count], 'geometry': [geom1, geom2] })], ignore_index=True))

matchingLayer['id'] = range(len(matchingLayer))
matchingLayer.set_crs(epsg=2154, inplace=True)
matchingLayer.explore(column='id', cmap='tab20', control_scale=True)